# 06 — Comparative Evaluation: 4 Model di Gold-Test

Evaluasi BERT, RoBERTa, DistilBERT, Detoxify pada `data/gold/test.csv` (held-out 30%) untuk sentimen 3-class dan toksisitas multi-label.

Output:
- `reports/eval_sentiment.csv` (4 model × metrik dengan bootstrap CI 95%)
- `reports/eval_toxicity.csv`
- `reports/confusion_<model>_<task>.{png,csv}`
- `reports/error_samples_<model>_<task>.csv`
- `reports/jargon_error_breakdown.csv`
- `reports/comparison_summary.md`

In [1]:
%pip install tabulate
!pip install tabulate


Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python310\\Scripts\\tabulate.exe' -> 'c:\\Python310\\Scripts\\tabulate.exe.deleteme'


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Sel 1: Setup
import sys
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'src').is_dir() and (p / 'configs').is_dir()), _here)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import os
os.chdir(_root)

from src.runtime import load_config, print_banner, RunLog
config = load_config('configs/experiment.yaml')
print_banner('06_comparative_evaluation', config)
run_log = RunLog(notebook='06_comparative_evaluation', config_path='configs/experiment.yaml')

import pandas as pd
import numpy as np
GOLD_ROOT = Path(config['data']['gold_root'])
INF_ROOT = Path(config['data']['inference_root'])
REPORTS = Path('reports')
REPORTS.mkdir(exist_ok=True)
(REPORTS / 'plots').mkdir(exist_ok=True)

SENT_LABELS = config['labels']['sentiment_classes']
TOX_LABELS = config['labels']['toxicity_labels']
MODELS = ['bert', 'roberta', 'distilbert']
BOOT_N = int(config['evaluation']['bootstrap_resamples'])
BOOT_SEED = int(config['seed'])

c:\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook: 06_comparative_evaluation
Experiment: thesis-sentiment-toxicity-dota2-decade
Seed: 42  |  Git: be03038
Started at: 2026-05-05T06:50:19+00:00Z
Versi paket:
  - python: 3.10.0
  - transformers: 5.5.0
  - torch: 2.8.0+cu128
  - datasets: 4.8.5
  - scikit-learn: 1.7.1
  - pandas: 2.3.3
  - numpy: 1.26.4


In [3]:
# Sel 2: Load gold-test + concat seluruh inferensi (subset baris yang ada di test)
test = pd.read_csv(GOLD_ROOT / 'test.csv')
print(f'Gold test: {len(test):,} baris')
test['key_tuple'] = list(zip(test['match_id'], test['time'], test['player_slot']))

def load_inference_for_test(folder: str) -> pd.DataFrame | None:
    inf_root = INF_ROOT / folder
    if not inf_root.exists():
        return None
    frames = [pd.read_parquet(p) for p in sorted(inf_root.glob('*.parquet'))]
    if not frames:
        return None
    inf = pd.concat(frames, ignore_index=True)
    inf['key_tuple'] = list(zip(inf['match_id'], inf['time'], inf['player_slot']))
    return inf

test_keys = set(test['key_tuple'])
print(f'Memfilter inferensi ke {len(test_keys):,} kunci test...')

Gold test: 2,392 baris
Memfilter inferensi ke 2,392 kunci test...


In [4]:
# Sel 3: Evaluasi sentimen
from src.eval.metrics import compute_sentiment_metrics
from src.eval.bootstrap_ci import bootstrap_ci
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

label_to_idx = {n: i for i, n in enumerate(SENT_LABELS)}
y_true_test = np.array([label_to_idx.get(s, -1) for s in test['sentiment']])
valid_mask = y_true_test >= 0

rows_sent = []
for model_key in MODELS:
    inf = load_inference_for_test(f'{model_key}_sentiment')
    if inf is None:
        msg = f'Skip eval sentiment {model_key} — inferensi tidak ada'
        print(f'[WARN] {msg}'); run_log.add_warning(msg); continue
    sub = inf[inf['key_tuple'].isin(test_keys)].copy()
    # Dedup: inference parquet bisa punya duplicate (match_id, time, player_slot)
    # karena sample.csv asli punya 31 dup yang lolos filter. Drop dulu sebelum merge.
    sub = sub.drop_duplicates(subset='key_tuple', keep='first')
    merged = test.merge(sub, on='key_tuple', how='left', suffixes=('', '_inf'))
    assert len(merged) == len(test), f'merge inflate row: {len(merged)} vs {len(test)}'
    pred_labels = merged['predicted_label'].map(label_to_idx).fillna(-1).astype(int).values
    prob_cols = [f'prob_{n}' for n in SENT_LABELS if f'prob_{n}' in merged.columns]
    y_score = merged[prob_cols].values if prob_cols else None
    mask = (y_true_test >= 0) & (pred_labels >= 0)
    metrics = compute_sentiment_metrics(
        y_true_test[mask], pred_labels[mask], SENT_LABELS,
        y_score=y_score[mask] if y_score is not None else None,
    )
    # Bootstrap CI untuk akurasi + F1-macro
    yt, yp = y_true_test[mask], pred_labels[mask]
    acc_mean, acc_lo, acc_hi = bootstrap_ci(accuracy_score, yt, yp, n_resamples=BOOT_N, seed=BOOT_SEED)
    f1m_mean, f1m_lo, f1m_hi = bootstrap_ci(
        lambda a, b: f1_score(a, b, average='macro', zero_division=0),
        yt, yp, n_resamples=BOOT_N, seed=BOOT_SEED,
    )
    row = {
        'model': model_key,
        'n_test': int(mask.sum()),
        'accuracy': metrics.accuracy,
        'accuracy_ci_lo': acc_lo, 'accuracy_ci_hi': acc_hi,
        'f1_macro': metrics.f1_macro,
        'f1_macro_ci_lo': f1m_lo, 'f1_macro_ci_hi': f1m_hi,
        'precision_macro': metrics.precision_macro,
        'recall_macro': metrics.recall_macro,
        'roc_auc_ovr': metrics.roc_auc_ovr,
    }
    for cls in SENT_LABELS:
        row[f'f1_{cls}'] = metrics.f1_per_class.get(cls, float('nan'))
    rows_sent.append(row)
    pd.DataFrame(metrics.confusion, index=SENT_LABELS, columns=SENT_LABELS).to_csv(
        REPORTS / f'confusion_{model_key}_sentiment.csv'
    )

# Detoxify zero-shot mapping ke 3-class sentiment
inf_det = load_inference_for_test('detoxify_toxicity')
if inf_det is not None:
    sub = inf_det[inf_det['key_tuple'].isin(test_keys)].copy()
    sub = sub.drop_duplicates(subset='key_tuple', keep='first')
    merged = test.merge(sub, on='key_tuple', how='left')
    assert len(merged) == len(test), f'merge inflate row: {len(merged)} vs {len(test)}'
    pred_labels = np.where(
        merged['max_toxicity_prob'].fillna(0) >= 0.5,
        label_to_idx['negative'],
        label_to_idx['neutral'],
    )
    mask = y_true_test >= 0
    metrics = compute_sentiment_metrics(y_true_test[mask], pred_labels[mask], SENT_LABELS)
    f1_per_class = dict(metrics.f1_per_class)
    f1_per_class['positive'] = float('nan')
    valid_f1s = [v for k, v in f1_per_class.items() if not (isinstance(v, float) and np.isnan(v))]
    f1_macro_2class = float(np.mean(valid_f1s)) if valid_f1s else float('nan')
    rows_sent.append({
        'model': 'detoxify_zero_shot',
        'n_test': int(mask.sum()),
        'accuracy': metrics.accuracy,
        'accuracy_ci_lo': float('nan'), 'accuracy_ci_hi': float('nan'),
        'f1_macro': f1_macro_2class,
        'f1_macro_ci_lo': float('nan'), 'f1_macro_ci_hi': float('nan'),
        'precision_macro': metrics.precision_macro,
        'recall_macro': metrics.recall_macro,
        'roc_auc_ovr': metrics.roc_auc_ovr,
        'note': 'zero-shot, tidak prediksi `positive` — f1_positive=NaN, f1_macro 2-class only',
        **{f'f1_{cls}': f1_per_class[cls] for cls in SENT_LABELS},
    })

eval_sent = pd.DataFrame(rows_sent)
eval_sent.to_csv(REPORTS / 'eval_sentiment.csv', index=False)
print(eval_sent.to_string(index=False))
run_log.add_output(REPORTS / 'eval_sentiment.csv')

             model  n_test  accuracy  accuracy_ci_lo  accuracy_ci_hi  f1_macro  f1_macro_ci_lo  f1_macro_ci_hi  precision_macro  recall_macro  roc_auc_ovr  f1_negative  f1_neutral  f1_positive                                                                          note
              bert    2392  0.388378        0.369147        0.408027  0.207978        0.192260        0.228921         0.243652      0.315051     0.593491     0.026316    0.551171     0.046448                                                                           NaN
           roberta    2392  0.395903        0.376662        0.415980  0.221881        0.196769        0.253007         0.272092      0.342617     0.517719     0.068966    0.561564     0.035112                                                                           NaN
        distilbert    2392  0.395067        0.375826        0.415144  0.217435        0.196776        0.244388         0.264582      0.331024     0.473103     0.050000    0.559338     0.0

In [5]:
# Sel 4: Evaluasi toksisitas (multi-label)
from src.eval.metrics import compute_toxicity_metrics
from sklearn.metrics import f1_score, hamming_loss

tox_label_cols = [f'tox_{lbl}' for lbl in TOX_LABELS]
y_tox_true = test[tox_label_cols].astype(int).values  # (n, 6)

rows_tox = []
for model_key in MODELS + ['detoxify']:
    inf = load_inference_for_test(f'{model_key}_toxicity')
    if inf is None:
        msg = f'Skip eval toxicity {model_key} — inferensi tidak ada'
        print(f'[WARN] {msg}'); run_log.add_warning(msg); continue
    sub = inf[inf['key_tuple'].isin(test_keys)].copy()
    sub = sub.drop_duplicates(subset='key_tuple', keep='first')
    merged = test.merge(sub, on='key_tuple', how='left')
    assert len(merged) == len(test), f'merge inflate row: {len(merged)} vs {len(test)}'
    pred_cols = [f'pred_{lbl}' for lbl in TOX_LABELS]
    prob_cols = [f'prob_{lbl}' for lbl in TOX_LABELS]
    y_pred = merged[pred_cols].fillna(0).astype(int).values
    y_score = merged[prob_cols].fillna(0.0).values if all(c in merged.columns for c in prob_cols) else None
    metrics = compute_toxicity_metrics(y_tox_true, y_pred, TOX_LABELS, y_score=y_score)
    f1mi_mean, f1mi_lo, f1mi_hi = bootstrap_ci(
        lambda a, b: f1_score(a, b, average='micro', zero_division=0),
        y_tox_true, y_pred, n_resamples=BOOT_N, seed=BOOT_SEED,
    )
    f1ma_mean, f1ma_lo, f1ma_hi = bootstrap_ci(
        lambda a, b: f1_score(a, b, average='macro', zero_division=0),
        y_tox_true, y_pred, n_resamples=BOOT_N, seed=BOOT_SEED,
    )
    row = {
        'model': model_key,
        'n_test': len(merged),
        'f1_micro': metrics.f1_micro,
        'f1_micro_ci_lo': f1mi_lo, 'f1_micro_ci_hi': f1mi_hi,
        'f1_macro': metrics.f1_macro,
        'f1_macro_ci_lo': f1ma_lo, 'f1_macro_ci_hi': f1ma_hi,
        'hamming_loss': metrics.hamming_loss,
        'subset_accuracy': metrics.subset_accuracy,
    }
    for lbl in TOX_LABELS:
        row[f'f1_{lbl}'] = metrics.f1_per_label.get(lbl, float('nan'))
        row[f'ap_{lbl}'] = metrics.avg_precision_per_label.get(lbl, float('nan'))
    rows_tox.append(row)

eval_tox = pd.DataFrame(rows_tox)
eval_tox.to_csv(REPORTS / 'eval_toxicity.csv', index=False)
print(eval_tox.to_string(index=False))
run_log.add_output(REPORTS / 'eval_toxicity.csv')

c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
c:\Python310\lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thr

     model  n_test  f1_micro  f1_micro_ci_lo  f1_micro_ci_hi  f1_macro  f1_macro_ci_lo  f1_macro_ci_hi  hamming_loss  subset_accuracy  f1_toxic  ap_toxic  f1_severe_toxic  ap_severe_toxic  f1_obscene  ap_obscene  f1_threat  ap_threat  f1_insult  ap_insult  f1_identity_hate  ap_identity_hate
      bert    2392  0.125000        0.019995        0.247843  0.074897        0.014493        0.140760      0.005853         0.979097  0.075472  0.047068              0.0              0.0    0.200000    0.060991        0.0        0.0   0.173913   0.075398               0.0               0.0
   roberta    2392  0.141414        0.031994        0.258655  0.076812        0.017778        0.133943      0.005923         0.979933  0.120000  0.067796              0.0              0.0    0.260870    0.142459        0.0        0.0   0.080000   0.093287               0.0               0.0
distilbert    2392  0.128205        0.020828        0.256436  0.068257        0.008125        0.133045      0.004738        

In [6]:
# Sel 5: Confusion matrix plots + error samples + jargon breakdown
import matplotlib.pyplot as plt
import seaborn as sns
from src.eval.error_analysis import sample_errors_sentiment, sample_errors_toxicity, jargon_error_breakdown

for model_key in MODELS:
    cm_path = REPORTS / f'confusion_{model_key}_sentiment.csv'
    if not cm_path.exists():
        continue
    cm = pd.read_csv(cm_path, index_col=0)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'{model_key} — sentiment')
    plt.tight_layout()
    plt.savefig(REPORTS / f'confusion_{model_key}_sentiment.png', dpi=120)
    plt.close(fig)

# Error samples (sentiment)
for model_key in MODELS:
    inf = load_inference_for_test(f'{model_key}_sentiment')
    if inf is None:
        continue
    sub = inf[inf['key_tuple'].isin(test_keys)].copy()
    sub = sub.drop_duplicates(subset='key_tuple', keep='first')
    merged = test.merge(sub, on='key_tuple', how='left')
    err = sample_errors_sentiment(merged, label_col='sentiment', pred_col='predicted_label', n_per_class=50)
    err.to_csv(REPORTS / f'error_samples_{model_key}_sentiment.csv', index=False)

# Jargon breakdown
rows = []
for model_key in MODELS:
    inf = load_inference_for_test(f'{model_key}_sentiment')
    if inf is None:
        continue
    sub = inf[inf['key_tuple'].isin(test_keys)].copy()
    sub = sub.drop_duplicates(subset='key_tuple', keep='first')
    merged = test.merge(sub, on='key_tuple', how='left')
    if 'is_dota_jargon' not in merged.columns:
        merged['is_dota_jargon'] = 0
    breakdown = jargon_error_breakdown(merged, label_col='sentiment', pred_col='predicted_label')
    breakdown['model'] = model_key
    breakdown['task'] = 'sentiment'
    rows.append(breakdown)
if rows:
    pd.concat(rows, ignore_index=True).to_csv(REPORTS / 'jargon_error_breakdown.csv', index=False)
    run_log.add_output(REPORTS / 'jargon_error_breakdown.csv')

In [7]:
# Sel 6: comparison_summary.md siap-kutip
lines = ['# Ringkasan Perbandingan 4 Model\n', '## Sentimen 3-class\n']
if not eval_sent.empty:
    cols = ['model', 'accuracy', 'accuracy_ci_lo', 'accuracy_ci_hi', 'f1_macro', 'f1_macro_ci_lo', 'f1_macro_ci_hi'] + [f'f1_{c}' for c in SENT_LABELS]
    cols = [c for c in cols if c in eval_sent.columns]
    lines.append(eval_sent[cols].to_markdown(index=False, floatfmt='.3f'))
lines.append('\n## Toksisitas Multi-Label\n')
if not eval_tox.empty:
    cols = ['model', 'f1_micro', 'f1_micro_ci_lo', 'f1_micro_ci_hi', 'f1_macro', 'hamming_loss', 'subset_accuracy'] + [f'f1_{l}' for l in TOX_LABELS]
    cols = [c for c in cols if c in eval_tox.columns]
    lines.append(eval_tox[cols].to_markdown(index=False, floatfmt='.3f'))

(REPORTS / 'comparison_summary.md').write_text('\n'.join(lines), encoding='utf-8')
print(f'Tertulis: {REPORTS / "comparison_summary.md"}')
run_log.add_output(REPORTS / 'comparison_summary.md')
run_log.save('reports/run_log.csv')

Tertulis: reports\comparison_summary.md
[run_log] 06_comparative_evaluation → 36.15s, 4 outputs, 0 warnings → reports\run_log.csv
